# Synthetic end-to-end validation — abundance recovery (5.02)R/ggplot figures for the classifier-filter → salmon pipeline on the synthetic chr1L1-insertion benchmark. Consumes `results/synthetic_validation/abundance.csv`(per L1 element, per grid cell: `simulated` insertion count vs `albertsalmon_seqlabel`salmon NumReads) and writes R² tables + figures.- Figures → `reports/figures/synthetic_validation/`- R² tables → `results/synthetic_validation/`

In [ ]:
library(data.table)library(ggplot2)# Okabe-Ito (colorblind-safe)OKABE_ITO <- c("#E69F00", "#56B4E9", "#009E73", "#F0E442",               "#0072B2", "#D55E00", "#CC79A7", "#000000")theme_set(theme_bw(base_size = 14))results_dir <- file.path("..", "results", "synthetic_validation")figures_dir <- file.path("..", "reports", "figures", "synthetic_validation")dir.create(figures_dir, recursive = TRUE, showWarnings = FALSE)# png + pdf at 300 dpi (style guide); white backgroundsave_fig <- function(plot, stem, width, height) {  for (ext in c("png", "pdf")) {    suppressMessages(ggsave(file.path(figures_dir, paste0(stem, ".", ext)),                            plot, width = width, height = height, dpi = 300, bg = "white"))  }  invisible(plot)}

In [ ]:
# --- load abundance table --------------------------------------------------ab <- fread(file.path(results_dir, "abundance.csv"))ab[, power := as.integer(power)]ab[, insertions := 2L^power]cat(sprintf("rows=%d  cells=%d  elements=%d  subfamilies=%d\n",            nrow(ab), uniqueN(ab[, .(power, del_prob)]),            uniqueN(ab$l1_id), uniqueN(ab$subfamily)))head(ab)

In [ ]:
# --- R^2 of AlbertSalmon vs Simulated --------------------------------------r2 <- function(x, y) {  if (length(x) < 3 || sd(x) == 0 || sd(y) == 0) return(NA_real_)  summary(lm(y ~ x))$r.squared}r2_by_level <- ab[, .(r2 = r2(simulated, albertsalmon_seqlabel)), by = power][order(power)]fwrite(r2_by_level, file.path(results_dir, "r2_by_insertion_rate.csv"))r2_by_del <- ab[, .(r2 = r2(simulated, albertsalmon_seqlabel)), by = del_prob][order(del_prob)]fwrite(r2_by_del, file.path(results_dir, "r2_by_del_probability.csv"))overall_r2 <- r2(ab$simulated, ab$albertsalmon_seqlabel)print(r2_by_level); print(r2_by_del)cat(sprintf("overall per-element R^2 = %.3f\n", overall_r2))

In [ ]:
# --- Figure: abundace_by_insertion_rate (R^2 vs insertion level) -----------fig1 <- ggplot(r2_by_level, aes(power, r2)) +  geom_line(colour = OKABE_ITO[5], linewidth = 1) +  geom_point(colour = OKABE_ITO[5], size = 2.5) +  scale_x_continuous(breaks = 5:13, labels = function(p) parse(text = paste0("2^", p))) +  coord_cartesian(ylim = c(0, 1)) +  labs(x = "Insertion level", y = expression(R^2 ~ "(AlbertSalmon vs Simulated)")) +  theme(panel.grid.minor = element_blank())save_fig(fig1, "abundace_by_insertion_rate", 6, 4)fig1

In [ ]:
# --- Figure: synthetic_albertsalmon_scatter (per-element, pooled) ----------fig2 <- ggplot(ab, aes(simulated, albertsalmon_seqlabel)) +  geom_point(alpha = 0.25, size = 0.7, colour = OKABE_ITO[2]) +  geom_smooth(method = "lm", se = TRUE, colour = OKABE_ITO[6], fill = OKABE_ITO[6]) +  annotate("text", x = -Inf, y = Inf, hjust = -0.15, vjust = 1.5,           label = sprintf("R^2 == %.3f", overall_r2), parse = TRUE, size = 5) +  labs(x = "Simulated abundance (insertions)", y = "AlbertSalmon (salmon NumReads)") +  theme(panel.grid.minor = element_blank())save_fig(fig2, "synthetic_albertsalmon_scatter", 5, 5)fig2

In [ ]:
# --- Figure: per-subfamily aggregated R^2 ----------------------------------ab_sub <- ab[, .(simulated = sum(simulated),                 albertsalmon_seqlabel = sum(albertsalmon_seqlabel)),             by = .(power, del_prob, subfamily)]r2_sub <- ab_sub[, .(r2 = r2(simulated, albertsalmon_seqlabel), n_elem = .N),                 by = subfamily][order(-r2)]fwrite(r2_sub, file.path(results_dir, "r2_by_subfamily.csv"))fig3 <- ggplot(r2_sub[!is.na(r2)], aes(reorder(subfamily, r2), r2)) +  geom_col(fill = OKABE_ITO[3]) +  coord_flip(ylim = c(0, 1)) +  labs(x = NULL, y = expression(R^2 ~ "(per-subfamily, aggregated over cells)")) +  theme(panel.grid.minor = element_blank())save_fig(fig3, "synthetic_albertsalmon_by_subfamily", 6, 4.5)fig3